# v3.5 EfficientSAM ONNX Export and Runtime

Exports EfficientViT-SAM-L0 decoder to ONNX and runs via onnxruntime CPU.

**License:** Apache-2.0 (EfficientViT-SAM)  
**Checkpoint:** ~/.cache/visionservex/efficientsam/efficientvit_sam_l0.pt  
**New in v3.5:** First working ONNX export of EfficientSAM (v3.4 was blocked by missing img_size attribute)

In [ ]:
import pathlib, json, warnings
import numpy as np
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/efficientsam_onnx_result.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Run export first: see code below')

In [ ]:
# EfficientSAM ONNX Export (requires checkpoint at ~/.cache/visionservex/efficientsam/)
from efficientsam.sam_model_zoo import create_efficientvit_sam_model
from efficientsam.segment_anything.utils.onnx import SamOnnxModel
import torch, warnings, pathlib
warnings.filterwarnings('ignore')

ckpt = str(pathlib.Path.home() / '.cache/visionservex/efficientsam/efficientvit_sam_l0.pt')
sam = create_efficientvit_sam_model('efficientvit-sam-l0', weight_url=ckpt)
sam.eval()
sam.image_encoder.img_size = 512  # Patch: EfficientViT encoder lacks img_size attr

embed_dim = sam.prompt_encoder.embed_dim
embed_size = sam.prompt_encoder.image_embedding_size
mask_input_size = [4 * x for x in embed_size]

onnx_model = SamOnnxModel(sam, return_single_mask=True)
dummy = {
    'image_embeddings': torch.randn(1, embed_dim, *embed_size),
    'point_coords': torch.randint(0, 512, (1, 5, 2), dtype=torch.float),
    'point_labels': torch.randint(0, 4, (1, 5), dtype=torch.float),
    'mask_input': torch.zeros(1, 1, *mask_input_size),
    'has_mask_input': torch.tensor([1.0]),
    'orig_im_size': torch.tensor([512.0, 512.0]),
}
out_path = 'efficientsam_l0.onnx'
torch.onnx.export(onnx_model, tuple(dummy.values()), out_path,
    opset_version=17, input_names=list(dummy.keys()),
    output_names=['masks','iou_predictions','low_res_masks'],
    dynamic_axes={'point_coords':{1:'pts'},'point_labels':{1:'pts'}}, dynamo=False)
print(f'Exported: {pathlib.Path(out_path).stat().st_size/1e6:.1f} MB')

## Evidence

Artifact: `notebook/99_final_report/artifacts/v35/efficientsam_l0_decoder.onnx` (16.5 MB, opset 17)  
Decoder latency: 11.07ms CPU  
Mask shape: [1, 1, 512, 512]  
IoU pred: 0.5752